# Лабораторная №6. Представления узлов графа и классификация вершин

__Автор задач: Пасканов В.Д.__

Цель: сравнить качество решения задачи классификации вершин графа для трёх подходов:

1. Бейзлайн без графовых признаков (например, CatBoost по табличным признакам).
2. Node2Vec с использованием готовой библиотеки.
3. Кастомный метод обхода графа + выбранная модель из NLP.


Материалы:

* Документация:
  * Node2Vec (реализация на Python): https://github.com/eliorc/node2vec
  * Gensim Word2Vec: https://radimrehurek.com/gensim/models/word2vec.html
  * CatBoost: https://catboost.ai/en/docs/
  * PyTorch: https://pytorch.org/docs/stable/index.html
  * HuggingFace Transformers (для идей по архитектурам): https://huggingface.co/docs/transformers/index

* Обзоры методов обхода графа и random walk:
  * Node2Vec: Scalable Feature Learning for Networks (Grover, Leskovec, 2016): https://snap.stanford.edu/node2vec/
  * DeepWalk: Online Learning of Social Representations (Perozzi et al., 2014): https://arxiv.org/abs/1403.6652
  * Role2Vec: https://arxiv.org/abs/1802.02896
  * Biased random walks для node2vec (обзор): https://memgraph.com/blog/how-node2vec-works


## Задачи для самостоятельного решения


### Задание 1. Выбор датасета для классификации узлов

Выберите **графовый датасет для задачи классификации узлов**, в котором:

* число узлов не менее 1000;
* у узлов есть **метки классов** (labels);
* у узлов желательно есть **атрибуты/признаки** (feature matrix), чтобы можно было построить бейзлайн без использования структуры графа.

Примеры источников графовых датасетов:

* PyTorch Geometric Datasets (Cora, Citeseer, Pubmed, Amazon, Reddit и др.):
  * https://pytorch-geometric.readthedocs.io/en/latest/modules/datasets.html
* SNAP (Stanford Network Analysis Project):
  * https://snap.stanford.edu/data/index.html
* Open Graph Benchmark (OGB):
  * https://ogb.stanford.edu/docs/nodeprop/

Рекомендуется взять один из классических датасетов (Cora / Citeseer / Pubmed / OGBN-Arxiv и т.п.), либо другой датасет, который удовлетворяет требованиям по размеру и наличию меток.

Опишите в этом ноутбуке, какой датасет вы выбрали:

* краткое текстовое описание (что за граф, что означают узлы и рёбра);
* число узлов, число рёбер, число классов;
* какие признаки есть у узлов.


### Задание 2. Train/val/test сплит по узлам

Сделайте разбиение узлов на три непересекающихся подмножества:

* `train` — для обучения моделей;
* `val` — для подбора гиперпараметров;
* `test` — для финальной оценки качества.

Рекомендуемые пропорции: 60% / 20% / 20% от числа узлов.

Желательно, чтобы распределение классов в каждом сплите было примерно одинаковым (стратифицированное разбиение по меткам).


### Задание 3. Бейзлайн модель без графа (CatBoost)

Постройте бейзлайн, который **не использует структуру графа**, а работает только с табличными признаками узлов.
В качестве такой модели можно использовать, например, `CatBoostClassifier`.

Шаги:

1. Подготовьте матрицу признаков `X` для узлов и вектор меток `y`.
2. Разделите `X` и `y` на `X_train`, `X_val`, `X_test` согласно сплиту по узлам.
3. Обучите модель CatBoost на `train`, подберите гиперпараметры по `val` (или используйте разумные значения по умолчанию).
4. Оцените качество на `test` (accuracy, F1, macro-F1 и др.).

Сохраните метрики бейзлайна — они понадобятся для сравнения с графовыми методами.


### Задание 4. Node2Vec с помощью библиотеки `node2vec`

Теперь построим представления узлов графа с помощью алгоритма Node2Vec.

Шаги:

1. Постройте граф `G` в формате `networkx.Graph` или `networkx.DiGraph`.
2. Используйте библиотеку `node2vec` (https://github.com/eliorc/node2vec) для генерации biased random walks и обучения эмбеддингов.
3. Получите матрицу эмбеддингов `Z` (num_nodes x embedding_dim).
4. Постройте модель классификации узлов по эмбеддингам (логрег / MLP / CatBoost).
5. Оцените качество на `test` и сохраните метрики.


### Задание 5. Модификация обхода графа

В этом задании вам необходимо **модифицировать стратегию обхода графа** (random walk), чтобы улучшить качество генерируемых траекторий для обучения эмбеддингов.

Возможные направления модификаций:

* Использовать **biased random walks**, как в Node2Vec (параметры $p$ и $q$ управляют склонностью гулять локально или делать переходы "подальше").
* Делать переходы с вероятностями, зависящими от **степени** вершины или её **PageRank**.
* Вводить вероятности телепортации (random walk with restart).
* Использовать информацию о классах/кластеризации для более частого посещения похожих вершин.

Полезные статьи/источники для вдохновения:

* Node2Vec: Scalable Feature Learning for Networks — идея biased random walks: https://snap.stanford.edu/node2vec/
* DeepWalk: Online Learning of Social Representations — классические случайные обходы: https://arxiv.org/abs/1403.6652
* Role2Vec: Node embeddings для ролей (role-based random walks): https://arxiv.org/abs/1802.02896
* Обзор random walk методов и их связи с PageRank: https://link.springer.com/chapter/10.1007/978-3-319-17290-3_1

Вам нужно:

1. Реализовать свою функцию генерации траекторий `generate_custom_walks(G, ...)`, которая возвращает коллекцию последовательностей узлов.
2. Обосновать, почему выбранная стратегия может давать более полезные траектории, чем простой случайный обход.
3. Использовать эти траектории для обучения новой модели эмбеддингов.


### Задание 6. Выбор модели из NLP для обучения на траекториях

Используя траектории узлов, полученные в задании 5, выберите **модель из NLP**, чтобы обучить эмбеддинги узлов.

Возможные варианты моделей:

1. **Word2Vec / Skip-gram / CBOW** (Gensim):
   * https://radimrehurek.com/gensim/models/word2vec.html
   * Варианты: изменить размерность, окно, negative sampling, sg=0/1.

2. **FastText** (Gensim) — учитывает подслова, можно попробовать, если вы хотите более "гладкие" эмбеддинги:
   * https://radimrehurek.com/gensim/models/fasttext.html

3. **Простая sequence-модель на PyTorch** (RNN / GRU / LSTM / Transformer):
   * идея: обучить модель предсказывать следующую вершину в траектории,
     а внутренние представления скрытого слоя/энкодера использовать как эмбеддинги узлов.
   * полезные ссылки:
     * Туториал по языковому моделированию в PyTorch: https://pytorch.org/tutorials/beginner/transformer_tutorial.html
     * Примеры seq2seq моделей: https://pytorch.org/tutorials/

Требования к заданию:

* Чётко опишите, **какую модель** вы выбрали и **почему**.
* Обучите модель на траекториях из задания 5.
* Получите матрицу эмбеддингов узлов (например, усреднив по позициям, где узел встречается, или взяв эмбеддинг слоя embedding / encoder).
* Используйте эти эмбеддинги для решения задачи классификации узлов (аналогично шагу с Node2Vec).


### Задание 7. Сравнение метрик baseline, Node2Vec и кастомного метода

Сравните качество трёх подходов на тестовом множестве:

1. Бейзлайн без графа (CatBoost по исходным признакам).
2. Node2Vec + классификатор по эмбеддингам.
3. Кастомный обход + выбранная NLP-модель + классификатор по эмбеддингам.

Рекомендуется оформить результаты в виде таблицы и кратко их прокомментировать (1–2 абзаца текста).
